# 03 — Séries temporelles Sentinel-2

Construction de la table spatio-temporelle SeineCrops à partir des scènes
Sentinel-2 L2A cataloguées en sprint S1 (`02_disponibilite_s2.ipynb`).

Ce notebook couvre quatre étapes séquentielles :

1. Téléchargement de la bande SCL (60 m) et calcul de la disponibilité
   effective sur l'AOI (`f_valid_aoi`) pour chaque scène du catalogue dédupliqué.
2. Téléchargement des bandes spectrales retenues (B02, B04, B05, B06, B07,
   B08, B11) pour les scènes dont `f_valid_aoi ≥ 0.01` ; resampling à 10 m.
3. Calcul des indices (NDVI, EVI, NDWI, NDRE) et composite mensuel par médiane.
4. Agrégation zonale sur les parcelles RPG (mean, std, p10, p90) et chargement
   dans `derived.s2_parcelles_monthly` (PostGIS).

---

## AOI et tuiles

| Paire | Tuiles | Zone couverte |
|-------|--------|---------------|
| Nord  | 30UYA, 31UCR | Pays de Caux (rive droite) |
| Sud   | 30UYV, 31UCQ | Plateau de Neubourg (rive gauche) |

---

## Bandes et indices

| Bande / Indice | Type | Résolution native | Formule |
|----------------|------|-------------------|---------|
| B02 | Bande | 10 m | — |
| B04 | Bande | 10 m | — |
| B05 | Bande | 20 m → 10 m | — |
| B06 | Bande | 20 m → 10 m | — |
| B07 | Bande | 20 m → 10 m | — |
| B08 | Bande | 10 m | — |
| B11 | Bande | 20 m → 10 m | — |
| NDVI | Indice | — | (B08 − B04) / (B08 + B04) |
| EVI  | Indice | — | 2.5 × (B08 − B04) / (B08 + 6×B04 − 7.5×B02 + 1) |
| NDWI | Indice | — | (B08 − B11) / (B08 + B11) |
| NDRE | Indice | — | (B08 − B05) / (B08 + B05) |

Toutes les bandes à 20 m sont resamplées à 10 m (bilinéaire) avant calcul
des indices. Le composite est calculé par médiane mensuelle sur les scènes
dont `f_valid_aoi ≥ 0.01`.

---

## Feature set résultant

11 variables (7 bandes + 4 indices) × 4 statistiques (mean, std, p10, p90)
× 16 mois = **704 features** par parcelle, stockées dans
`derived.s2_parcelles_monthly`.

---

## Structure du notebook

| Section | Contenu |
|---------|---------|
| S3.1 | Téléchargement SCL et calcul `f_valid_aoi` — complétion du catalogue dédupliqué |
| S3.2 | Téléchargement bandes spectrales et calcul des indices — scènes retenues (`f_valid_aoi ≥ 0.01`) |
| S3.3 | Composite mensuel — médiane par mois × bande/indice × tuile |
| S3.4 | Agrégation zonale et chargement PostGIS — `derived.s2_parcelles_monthly` |

---

## Références

- [Documentation CDSE — téléchargement produits S2](https://documentation.dataspace.copernicus.eu/APIs/OData.html)
- [Sentinel-2 L2A — bande SCL (classes et usage)](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html)
- [HR-VPP — manuel produit trajectoires saisonnières (PDF)](https://land.copernicus.eu/en/technical-library/product-user-manual-of-seasonal-trajectories/@@download/file)
- `02_disponibilite_s2.ipynb` — sprint S1, catalogue CDSE, `df_dedup`, `AVAILABILITY_REPORT.json`
- `01_ingestion_rpg.ipynb` — sprint S1, `derived.rpg_parcelles_aoi` (géométries de référence)

### S3.1 — Téléchargement SCL et calcul `f_valid_aoi`

Téléchargement de la bande SCL (60 m) pour chaque scène du catalogue
dédupliqué (`df_dedup`, 4 tuiles, fenêtre sept. N → déc. N+1) et calcul
de la fraction de pixels valides sur l'AOI (`f_valid_aoi`).

`f_valid_aoi` est la métrique de disponibilité effective : contrairement au
`cloud_cover_catalogue` (couverture nuageuse déclarée sur la tuile entière),
elle mesure précisément la fraction de pixels exploitables **sur l'AOI**,
en excluant les classes SCL invalides :

| Classe SCL | Signification |
|------------|---------------|
| 3 | Ombres nuageuses |
| 8 | Nuages moyennement probables |
| 9 | Nuages hautement probables |
| 10 | Cirrus |
| 11 | Neige / glace |

Seules les scènes dont `f_valid_aoi ≥ 0.01` (au moins 1 % de pixels valides
sur l'AOI) sont retenues pour le téléchargement des bandes spectrales en S3.2.

Les fichiers SCL sont stockés dans `data/raw/s2/scl/<tile_id>/` et non
versionnés (`.gitignore`). La colonne `f_valid_aoi` est persistée dans
`data/raw/s2/catalogue_dedup.parquet` en clôture de cette section.

**Dépendance** : `df_dedup` produit par `02_disponibilite_s2.ipynb` —
relire `data/raw/s2/AVAILABILITY_REPORT.json` ou ré-exécuter nb02 si
le fichier Parquet est absent.